# 02 - Prepare mutations and labels

Turns the cleaned mutation table into one labelled row per patient, and pulls
the wild-type protein sequences that notebook 03 needs.

`assign_labels` supports three strategies and only one is defensible:

| strategy | what it does | verdict |
|---|---|---|
| `longitudinal` | outcome recorded at follow-up | the real task |
| `diagnosis` | AML/MDS sample = 1 | leaky - label is a function of the features |
| `chip_risk` | high-risk gene + VAF rule = 1 | circular - fitting your own rule |

Run the leaky ones once to prove the plumbing works, then never report them.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection")
    # Run every session, not only on first mount: Colab wipes packages on
    # restart, and misspelled package names fail silently without check=True.
    subprocess.run("pip install -q transformers accelerate lightgbm".split(), check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

CFG = yaml.safe_load(open("configs/config.yaml"))
CFG

In [ ]:
import json
from src.data_processing import aggregate_by_patient, assign_labels

mut = pd.read_parquet(f'{CFG["data"]["processed_dir"]}/mutations_clean.parquet')
clinical = pd.read_parquet(f'{CFG["data"]["processed_dir"]}/clinical.parquet')
patients = aggregate_by_patient(mut)
patients.head()

In [ ]:
strategy = CFG["labels"]["strategy"]
try:
    labelled = assign_labels(patients, strategy=strategy, clinical=clinical,
                             outcome_col=CFG["labels"]["outcome_col"])
except ValueError as e:
    print(f"'{strategy}' unavailable: {e}")
    print("Falling back to 'chip_risk' as a SMOKE TEST ONLY.")
    labelled = assign_labels(patients, strategy="chip_risk")

labelled[["patient_id", "n_mutations", "max_vaf", "label"]].head()

## Wild-type sequences

Fetched from UniProt once, then cached to `data/external/uniprot_cache.json`.
Commit that file - it makes the pipeline reproducible offline and avoids
hammering the API on every runtime restart.

In [ ]:
from src.sequences import get_sequences

genes_present = sorted(mut["gene"].unique())

# If notebook 01 ran on synthetic data, use the sequences it generated.
# Mixing synthetic mutations with real UniProt sequences gives a ~5% match
# rate (chance level) and a meaningless mismatch warning below.
syn_path = f'{CFG["data"]["processed_dir"]}/synthetic_sequences.json'
if os.path.exists(syn_path):
    sequences = {g: s for g, s in json.load(open(syn_path)).items()
                 if g in genes_present}
    print(f">>> SYNTHETIC sequences ({len(sequences)} genes). Not real proteins.")
else:
    sequences = get_sequences(genes_present, cache_path=CFG["data"]["uniprot_cache"])
    print(f"resolved {len(sequences)}/{len(genes_present)} genes from UniProt")

missing = set(genes_present) - set(sequences)
if missing:
    print("MISSING (no embeddings for these):", sorted(missing))


In [ ]:
# How many mutations survive to the embedding stage?
missense = mut[mut["kind"] == "missense"]
mapped = missense[missense["gene"].isin(sequences)]
print(f"{len(mut):,} mutations")
print(f"  {len(missense):,} missense")
print(f"  {len(mapped):,} with a sequence -> {mapped['mut_key'].nunique():,} unique to embed")

### Reference-residue agreement

Checks the wild-type amino acid in the MAF against UniProt. High mismatch means
the MAF was annotated on a different transcript - fix the transcript mapping,
because ESM2 cannot fix it for you.

Synthetic data generates its own sequences, so expect 0% there.

In [ ]:
ok = bad = 0
for _, r in mapped.iterrows():
    seq = sequences[r["gene"]]
    p = int(r["aa_pos"])
    if p <= len(seq) and seq[p - 1] == r["aa_wt"]:
        ok += 1
    else:
        bad += 1
total = max(ok + bad, 1)
print(f"reference match: {ok}/{total} ({ok/total:.1%})")
if bad / total > 0.15:
    print("!! >15% mismatch - transcript mapping problem, not a modelling problem.")

In [ ]:
labelled.to_parquet(f'{CFG["data"]["processed_dir"]}/patients_labelled.parquet')
print("saved patients_labelled.parquet")